# Decision Trees — Interpretable Models with Sharp Edges

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026F_predictive_analytics_QM474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2027S_predictive_analytics_QM474/blob/main/notebooks/nb11_decision_trees_student.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain how a decision tree partitions feature space using axis-aligned splits, for **both classification and regression** targets.
2. Fit and visualize a `DecisionTreeClassifier` (Wisconsin Breast Cancer) and a `DecisionTreeRegressor` (California Housing) using `plot_tree`.
3. Diagnose **overfitting** by comparing the training score with the 5-fold cross-validation score, and read the train–CV gap as a sign of memorization rather than as a verdict on held-out performance.
4. Compare a tree against the nb09 reference on identical CV folds with `compare_paired`: describe each model by its fold scores, mean and SD, and decide from the **corrected interval of the per-fold gain**, read against the margin band declared in advance (four possible outcomes).
5. Nominate a `max_depth` with the **one-standard-error heuristic** (SD/√5), then test the nominee against the depth you carry forward — a nomination is not a verdict.

---


> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises** — Exercise 1 on the classification track and Exercise 2 on the regression track. Complete both before submitting your notebook.

---

## 💼 Why This Matters

This notebook runs **two parallel cases** end-to-end, because your final-project group's target may be either classification or regression. Watching the same algorithm work on both cuts your transfer cost in half.

### The classification case — State Health Department screening

Wisconsin Breast Cancer biopsy data, 569 patients, 30 cell-nucleus measurements per biopsy, target = malignant vs benign. The Department's existing screening tool flags about 10% of clinic visits for confirmatory testing; the screening logistic-regression model from nb06 lifts true-positive rate but the oncologists push back: *"I can't explain to a patient why the model flagged them. I need to see the decision logic."*

A decision tree gives them a rule like the one you will fit in Section 4: **"if worst radius > 16.8, worst concavity > 0.22 and texture error > 0.45, then malignant."** The path from root to leaf is an auditable flowchart any clinician can read aloud. The MedScreen review board has one condition for a single-tree deliverable: the flowchart must fit on one page, so **at most three questions per path (`max_depth` ≤ 3)**.

### The regression case — HomeValue Analytics property listings

California Housing data, 20,640 census tracts, 8 features (median income, house age, occupancy, location), target = median house value in USD 100K units. HomeValue Analytics wants to deploy a price-prediction model on its public-facing listings: *"What is this property worth, given comparable tracts in the area?"*

A regression tree gives the legal team something they can defend in court: **"this property's prediction is the average of the comparable tracts that answered the same five questions."** No coefficients to interpret, no extrapolation outside the training range — just an averaging rule with a documented provenance. The legal team's condition for a single tree: **at most five questions per path (`max_depth` ≤ 5)** — beyond that, nobody reads the tree aloud in court.

### The reference every tree-based model has to beat

nb08 and nb09 closed with a disciplined comparison: tuned linear models against their defaults, scored on the **identical CV folds** and read with `compare_paired` — the per-fold gain, an interval corrected for the overlap between folds, and a margin fixed in advance. On both cases the tuned model was *practically equivalent* to the default, so the simpler model stayed:

- **Classification:** `Pipeline([StandardScaler, LogisticRegression(C=1.0, max_iter=5000)])` — tuning `C` bought nothing the review board would act on, so the default ships.
- **Regression:** `Pipeline([StandardScaler, LinearRegression()])` (OLS) — the best Ridge `α` was practically equivalent to OLS, so OLS ships.

Both stakeholders also declared, in nb08 and before any score was computed, the size of gain that would justify switching models — as a **band of three margins**: **base 0.01**, which decides, and **small 0.005** and **large 0.02**, which check whether the answer would change for a more relaxed or a more demanding stakeholder. The units are ROC-AUC for MedScreen (0.01 = one more correctly ordered malignant–benign pair in every hundred) and R² for HomeValue's CFO. Write the band down now: the same three numbers, unchanged, judge every head-to-head from here through nb14's selection ceremony.

Throughout nb11 → nb14 these two pipelines are the **nb09 reference models** — each stakeholder's current champion. A tree, forest, or boosting model replaces one only when it earns a **worthwhile gain**: the lower end of the corrected interval of its per-fold gain sits above the base margin. They appear as `reference_clf` and `reference_reg` in the setup cell and are the comparison floor in every model comparison from here through nb14's selection ceremony.

---


## 1. Setup — Imports, Plot Helpers, and the Comparison Tools

The setup cell imports both tree algorithms, registers two plot helpers (`plot_train_cv_curve`, `plot_predicted_vs_actual`), and pastes in the comparison tools you met in nb08: `compare_paired`, the two margin bands (`DELTAS_AUC`, `DELTAS_R2`), and three small helpers that print and plot a comparison. Locking the random seed to `RANDOM_SEED = 474` (course number) means every fold split, every tree fit, and every plot is reproducible across machines.


> 💡 **Gemini Prompt:** "Set up imports for sklearn DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, LogisticRegression, LinearRegression, cross_val_score, train_test_split, StratifiedKFold, KFold, load_breast_cancer, fetch_california_housing, and scipy.stats. Set RANDOM_SEED = 474. Define plot_train_cv_curve(x_values, train, cv_mean, cv_sd, xlabel, ylabel, title, ax) for overfitting diagnostics and plot_predicted_vs_actual(y_true, y_pred, ax, title) for regression diagnostics. Paste the course's compare_paired(scores_a, scores_b, margin) function from nb08 exactly as written, and define DELTAS_AUC and DELTAS_R2 as {'small': 0.005, 'base': 0.01, 'large': 0.02}."
>
> **After running, verify:**
> - [ ] `RANDOM_SEED = 474` printed back
> - [ ] `plot_train_cv_curve`, `plot_predicted_vs_actual` and `compare_paired` defined and callable
> - [ ] Both margin bands printed, with `base = 0.01`
> - [ ] No import errors


In [ ]:
# Setup — imports, seed, display, plot helpers, the paired comparison and the margin band
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import load_breast_cancer, fetch_california_housing, make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)

# --- Course color convention (nb19 alignment) ---
CLF_COLOR = '#1f77b4'   # blue — classification accent
REG_COLOR = '#ff7f0e'   # orange — regression accent
GREY      = '#999999'

# --- Helper 1: train-vs-CV overfitting curve (CV = the five held-out folds of X_train) ---
def plot_train_cv_curve(x_values, train, cv_mean, cv_sd, xlabel, ylabel, title, ax,
                        color_train=GREY, color_cv=CLF_COLOR):
    """Training score next to the 5-fold CV mean, with the fold SD as error bars (spread, not an interval)."""
    xs = list(range(len(x_values)))
    ax.plot(xs, train, marker='o', label='Train (all of X_train)',
            linewidth=2, color=color_train)
    ax.errorbar(xs, cv_mean, yerr=cv_sd, marker='s',
                label='5-fold CV mean ± fold SD', linewidth=2, capsize=5, color=color_cv)
    ax.set_xticks(xs)
    ax.set_xticklabels([str(v) for v in x_values])
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

# --- Helper 2: predicted-vs-actual scatter (the standard regression diagnostic) ---
def plot_predicted_vs_actual(y_true, y_pred, ax, title='Predicted vs Actual',
                             color=REG_COLOR):
    """Scatter of y_pred vs y_true with the y=x line of perfect prediction."""
    ax.scatter(y_true, y_pred, alpha=0.25, s=8, color=color)
    lo = float(min(np.min(y_true), np.min(y_pred)))
    hi = float(max(np.max(y_true), np.max(y_pred)))
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.5, label='Perfect prediction')
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

# --- compare_paired: the course's model-comparison rule from nb08 (pasted verbatim) ---
# A is the challenger, B the current champion. Both score arrays must come from ONE splitter object.
def compare_paired(scores_a, scores_b, margin):
    """Paired comparison on identical folds; higher score = better."""
    d = np.asarray(scores_a) - np.asarray(scores_b)
    k = len(d)
    t = stats.t.ppf(0.975, k - 1)
    se = d.std(ddof=1) * np.sqrt(1/k + 1/(k - 1))   # Nadeau–Bengio correction
    lo, hi = d.mean() - t*se, d.mean() + t*se
    if lo > margin:
        verdict = "A is better by a worthwhile amount"
    elif hi < -margin:
        verdict = "B is better by a worthwhile amount"
    elif -margin < lo and hi < margin:
        verdict = "Practically equivalent: choose the simpler model"
    else:
        verdict = "Unresolved: apply the default you declared"
    return d, d.mean(), (lo, hi), verdict

# --- The margin band each stakeholder declared in nb08, before any score was computed ---
# base decides the champion; small and large are the sensitivity check (same numbers through nb14).
DELTAS_AUC = {'small': 0.005, 'base': 0.01, 'large': 0.02}   # MedScreen review board — ROC-AUC
DELTAS_R2  = {'small': 0.005, 'base': 0.01, 'large': 0.02}   # HomeValue CFO — R²
DELTA_AUC  = DELTAS_AUC['base']
DELTA_R2   = DELTAS_R2['base']

def show_comparison(challenger, champion, scores_a, scores_b, deltas, metric):
    """Print one paired comparison: fold gains (description), corrected interval, verdict, band check."""
    d, gain, (lo, hi), verdict = compare_paired(scores_a, scores_b, deltas['base'])
    print(f"{challenger} (A, challenger) vs {champion} (B, current champion) — {metric}")
    print("  per-fold gains A − B (description only): " + ', '.join(f"{x:+.4f}" for x in d))
    print(f"  mean gain {gain:+.4f}; interval corrected for fold overlap [{lo:+.4f}, {hi:+.4f}]")
    print(f"  base margin δ = {deltas['base']}: {verdict}")
    band = {name: compare_paired(scores_a, scores_b, m)[3] for name, m in deltas.items()}
    same = len(set(v.startswith('A is better') for v in band.values())) == 1
    print("  band check: " + '; '.join(f"{n} {deltas[n]} → {v.split(':')[0]}" for n, v in band.items())
          + ("  → robust across the band" if same else "  → margin-sensitive"))
    return d, gain, (lo, hi), verdict

# --- Fold-score summary for ONE model: the five fold scores, their mean and their SD (no interval) ---
def fold_table(score_dict):
    rows = []
    for name, s in score_dict.items():
        s = np.asarray(s)
        rows.append({'model': name, **{f'fold {i+1}': v for i, v in enumerate(s)},
                     'mean': s.mean(), 'sd': s.std(ddof=1)})
    return pd.DataFrame(rows)

def plot_fold_scores(ax, score_dict, ylabel, title, colors=None):
    """Dot strip: each model's five fold scores and their mean — no error bars, no interval."""
    for j, (name, s) in enumerate(score_dict.items()):
        c = (colors or {}).get(name, GREY)
        ax.scatter(np.full(len(s), j) + np.linspace(-0.08, 0.08, len(s)), s, color=c, s=45,
                   edgecolor='black', zorder=3)
        ax.hlines(np.mean(s), j - 0.25, j + 0.25, color='black', linewidth=2, zorder=4)
    ax.set_xticks(range(len(score_dict)))
    ax.set_xticklabels(list(score_dict), fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

def plot_gain_intervals(ax, rows, margin, xlabel, title):
    """rows: list of (label, d, (lo, hi)). Dots = per-fold gains; bar = corrected interval; shaded = ±δ."""
    ax.axvspan(-margin, margin, color=GREY, alpha=0.15, label=f'±δ = ±{margin}')
    ax.axvline(0, color='black', linewidth=1)
    for j, (label, d, (lo, hi)) in enumerate(rows):
        ax.hlines(j, lo, hi, color='black', linewidth=3)
        ax.scatter(d, np.full(len(d), j), color=REG_COLOR if 'R²' in xlabel else CLF_COLOR,
                   s=35, edgecolor='black', zorder=3)
        ax.scatter([np.mean(d)], [j], marker='D', color='white', edgecolor='black', s=60, zorder=4)
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([r[0] for r in rows], fontsize=9)
    ax.set_xlabel(xlabel)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3, axis='x')

# --- nb09 reference models: each stakeholder's current champion (the incumbent every challenger faces) ---
# Classification:  default LogReg(C=1.0) — nb09's tuned C was practically equivalent to it, so the default stayed.
# Regression:      OLS                   — nb09's best Ridge was practically equivalent to it, so OLS stayed.
reference_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))
])
reference_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('reg',    LinearRegression())
])

print(f"✓ RANDOM_SEED = {RANDOM_SEED}")
print("✓ Helpers defined: plot_train_cv_curve, plot_predicted_vs_actual, compare_paired, "
      "show_comparison, fold_table, plot_fold_scores, plot_gain_intervals")
print(f"✓ Margin band (base decides): ROC-AUC {DELTAS_AUC}, R² {DELTAS_R2}")
print("✓ nb09 references defined: reference_clf (LogReg C=1.0), reference_reg (OLS)")
print("✓ Display precision: 4 digits")


**Reading the output:**

The setup cell does four jobs at once. First, it imports both `DecisionTreeClassifier` (for Wisconsin breast cancer) and `DecisionTreeRegressor` (for California housing) — the two algorithms we will pair-fit throughout the notebook. Second, it locks `RANDOM_SEED = 474` so every CV fold, every tree fit, and every plot reproduces identically across machines. Third, it defines two plot helpers: `plot_train_cv_curve` (the overfitting diagnostic you will see in Section 6, comparing the training score with the 5-fold CV score) and `plot_predicted_vs_actual` (the standard regression-diagnostic scatter, mandatory for every regression model on the final poster). Fourth, it brings in nb08's comparison tools unchanged: `compare_paired`, the margin bands, and `show_comparison`, which prints a comparison's per-fold gains, its corrected interval, its verdict at the base margin, and whether the verdict survives the small and large margins.

---


## 2. Decision Tree Intuition — A Flowchart You Can Read

A decision tree is the analytics version of a flowchart. At every step it asks one yes/no question about a single feature — *"is annual income above \$60K?"*, *"is the lump's worst radius greater than 16.8?"* — and routes the observation left or right. After a handful of questions every observation arrives at a **leaf**, and the leaf holds the answer: a predicted class for classification, an average value for regression. That is the whole algorithm — no coefficients to interpret, no equations to memorize, just a chain of yes/no questions ending in a verdict.

A fitted tree comes in two equivalent views and you will see both side by side below. The **flowchart view** is the boxes-and-branches diagram you would draw on a whiteboard for a non-technical colleague. The **partition view** is the same tree's predictions painted onto the feature space — the right framing for comparing what a tree does to what a logistic regression or linear regression would do on the same data. Both views describe the same fitted model; they just emphasize different things.

The picture below uses a synthetic two-feature dataset so the geometry is visible on a flat plane. The real datasets in this notebook live in 8 and 30 dimensions, but the rule is identical — one feature per question, axis-aligned cuts in the data plane, and a single constant prediction inside each rectangle.

In [ ]:
# Section 2: build intuition with a small tree and the partition it creates.
# A two-feature synthetic dataset lets us see the tree's geometry on a flat plane.
X_toy, y_toy = make_classification(
    n_samples=200, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=2, class_sep=0.7, flip_y=0.05,
    random_state=RANDOM_SEED
)

# A depth=2 tree is small enough that every box and leaf is legible at a glance.
demo_tree = DecisionTreeClassifier(max_depth=2, random_state=RANDOM_SEED).fit(X_toy, y_toy)

# Side-by-side: the flowchart view (left) and the partition it creates (right).
fig, (ax_tree, ax_plane) = plt.subplots(1, 2, figsize=(16, 6))

plot_tree(
    demo_tree,
    feature_names=['feature 1', 'feature 2'],
    class_names=['class A', 'class B'],
    filled=True,
    rounded=True,
    fontsize=11,
    ax=ax_tree,
)
ax_tree.set_title('Flowchart view — the fitted decision tree (max_depth=2)',
                  fontsize=13, fontweight='bold')

# Same tree, drawn on the 2D plane.
x_min, x_max = X_toy[:, 0].min() - 0.5, X_toy[:, 0].max() + 0.5
y_min, y_max = X_toy[:, 1].min() - 0.5, X_toy[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
Z = demo_tree.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax_plane.contourf(xx, yy, Z, alpha=0.25, cmap='RdBu')
ax_plane.scatter(X_toy[:, 0], X_toy[:, 1], c=y_toy, cmap='RdBu',
                 edgecolor='k', s=40)
ax_plane.set_xlabel('feature 1')
ax_plane.set_ylabel('feature 2')
ax_plane.set_title('Partition view — the same tree drawn on the data',
                   fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# Depth sweep: same dataset, deeper trees, finer partitions.
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, depth in zip(axes, [1, 2, 3, 6]):
    t = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_SEED).fit(X_toy, y_toy)
    Z = t.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdBu')
    ax.scatter(X_toy[:, 0], X_toy[:, 1], c=y_toy, cmap='RdBu',
               edgecolor='k', s=40)
    ax.set_xlabel('feature 1')
    ax.set_ylabel('feature 2')
    ax.set_title(f'max_depth = {depth}  →  {t.get_n_leaves()} leaves',
                 fontsize=12, fontweight='bold')

fig.suptitle('Deeper trees carve the plane into more rectangles — at depth 6 the tree is already memorizing points',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("✨ Each leaf in the flowchart on the left maps to one coloured rectangle on the right.")
print("✨ Every cut is horizontal or vertical — a tree never draws a diagonal line.")
print("✨ At depth 6 several leaves contain only one or two training points — that is overfitting in pictures.")


**Reading the output:**

Start with the flowchart on the left. The top box is the **root** — every training row enters here. The line inside the box (something like *"feature 2 <= 0.2"*) is the yes/no question this node asks. Rows that answer "yes" slide down the left branch; rows that answer "no" slide down the right. Each box also reports four pieces of information: `gini` is how mixed the classes are inside this box (0.0 means pure, 0.5 means a perfect 50/50 split), `samples` is how many training rows landed here, `value` is how those rows split between the two classes, and `class` is the box's current verdict. The boxes with no children — the bottom row — are the **leaves**, and once a row reaches a leaf the leaf's `class` is the tree's prediction.

Now look at the right panel. The plane is divided into four coloured rectangles, one per leaf in the flowchart. Pick any point in the plane, follow the questions in the tree, and you will land in the matching rectangle. That is what people mean when they call decision trees *interpretable*: the path from input to prediction is a short list of yes/no questions you could read aloud to a loan officer, a clinician, or a hiring manager and they would understand the decision without ever opening the model.

The four-panel figure below sweeps the depth lever on the same dataset. At `max_depth = 1` the tree asks one question, the plane is cut once, and many points sit on the wrong side of the line — that is **underfitting**. By `max_depth = 6` the tree has carved out so many tiny rectangles that some leaves contain only one or two training points, drawn around them like little fences — that is **overfitting**. Somewhere in the middle is a sweet spot, and finding it on real data is exactly what Sections 6 and 7 will quantify.

> **A question that often comes up here:** *"why are all the boundaries horizontal or vertical?"* Because each split asks about exactly one feature at a time, the cut it draws has to be perpendicular to that feature's axis. A logistic regression can draw a single diagonal line that uses both features at once; a tree has to approximate the same diagonal with a staircase of horizontal and vertical cuts. Trees can get there, but every step of the staircase costs depth — and depth is what makes a tree variance-hungry on small datasets, as the depth sweep on the right already hints at.

**Key takeaway:** A tree's depth is its bias-variance dial. Shallow trees ask too few questions and miss real signal (high bias). Deep trees ask so many questions that each leaf holds only a handful of training rows and the model starts memorizing them (high variance). The rest of this notebook is about finding the depth that balances the two on real data — not on this toy plane.

---

## 3. Load Both Datasets — One 80/20 Split, Two Locked Envelopes

Time to set up the actual data. Since nb08 the course splits each dataset **once**: **80% training / 20% test**, with the test envelope sealed until nb14. Every model fit, every diagnostic, every tuning decision between here and nb14 lives on the **training set** and is evaluated through 5-fold cross-validation — each fold takes a turn as the held-out sample, so every training row is used for fitting and for evaluation. There is no separate validation set any more: five folds do its job on more rows. The **test set** is the final blind exam the auditor grades once.

`random_state=RANDOM_SEED` keeps the split identical across runs, and it is the same first split nb08 and nb09 make, so the sealed test rows are the same ones in every notebook. Inside the training set, evaluation goes through **5-fold cross-validation**. The two tracks use slightly different CV splitters because the data shapes are different. Classification uses `StratifiedKFold`, which keeps each fold's mix of malignant and benign biopsies close to the overall \~63% benign / \~37% malignant balance — important whenever one class is the minority. Regression uses plain `KFold` because there are no classes to preserve; the housing target is a continuous price, and any random fold is already representative. Each splitter is created once and reused for every model on its case, which is what makes two models' fold scores comparable fold by fold.

The Wisconsin Breast Cancer data is small (569 patients, 30 features per biopsy) and California Housing is large (20,640 census tracts, 8 features per tract). That roughly 36× size gap will matter in the next sections — small datasets let trees overfit fast, large datasets tolerate deeper trees but slow CV down. Both datasets reappear in every comparison from here through Section 7.


> 💡 **Gemini Prompt:** "Apply the course's 80/20 split to both datasets. For the breast-cancer dataset use `train_test_split` with `test_size=0.20`, `random_state=474`, `stratify=y_clf` to produce `X_train_clf` / `X_test_clf`; for California Housing use the same call without stratification to produce `X_train_reg` / `X_test_reg`. Create `cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=474)` and `cv_reg = KFold(n_splits=5, shuffle=True, random_state=474)`. Print the size of each split and mark both test sets as LOCKED until nb14."
>
> **After running, verify:**
> - [ ] Classification: train 455, test 114 (out of 569)
> - [ ] Regression: train 16,512, test 4,128 (out of 20,640)
> - [ ] Both splits and both CV splitters use `random_state=RANDOM_SEED`, and both splitters use 5 folds
> - [ ] No validation set and no second split anywhere
> - [ ] Both `X_test_*` printouts include the LOCKED marker


In [ ]:
# --- Classification track: Wisconsin breast cancer (State Health Department) ---
data_clf = load_breast_cancer(as_frame=True)
X_clf = data_clf.data
y_clf = data_clf.target

# One 80/20 split (the pattern since nb08): X_train for all model work, X_test locked until nb14.
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_SEED, stratify=y_clf
)
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# --- Regression track: California Housing (HomeValue Analytics) ---
data_reg = fetch_california_housing(as_frame=True)
X_reg = data_reg.data
y_reg = data_reg.target  # target is median house value in units of USD 100K

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_SEED
)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print("=== CLASSIFICATION (State Health Department / Wisconsin Breast Cancer) ===")
print(f"  Features:  {X_clf.shape[1]}")
print(f"  Train:     {len(X_train_clf):>5} patients   (every model decision: 5-fold CV on these rows)")
print(f"  Test:      {len(X_test_clf):>5} patients   [LOCKED until nb14]")
print(f"  Class balance (train): {y_train_clf.mean():.3f} positive (benign)")

print()
print("=== REGRESSION (HomeValue Analytics / California Housing) ===")
print(f"  Features:  {X_reg.shape[1]}")
print(f"  Train:     {len(X_train_reg):>5} tracts     (every model decision: 5-fold CV on these rows)")
print(f"  Test:      {len(X_test_reg):>5} tracts     [LOCKED until nb14]")
print(f"  Target:    median house value, USD 100K units")
print(f"  y_train range: USD {y_train_reg.min()*100:.0f}K – USD {y_train_reg.max()*100:.0f}K, "
      f"mean USD {y_train_reg.mean()*100:.0f}K")


**Reading the output:**

One split per case, two locked envelopes, two CV plans. Notice the size gap. The Wisconsin biopsy data hands you 455 patients to train on; California Housing hands you 16,512 tracts. The housing set is roughly **36× larger**, and that asymmetry will shape every model decision in this notebook. With 455 patients and 30 features, a classification tree can memorize individual biopsies after just a handful of splits — small data, sharp risk of overfitting. Each CV fold holds out only about 91 patients, so fold scores on this case will bounce around more than you might expect. The regression tree has more than sixteen thousand tracts to learn from before that danger kicks in, so it tolerates deeper trees, and each of its folds holds out about 3,300 tracts.

The class-balance line on the classification side is worth a second look: 0.626 of the training biopsies are benign, so about 37% are malignant. That is the balance `StratifiedKFold` preserves in every CV fold. Without stratification, a random fold could land mostly benign cases and the model's accuracy would look misleadingly stable; with stratification, each fold sees the same realistic class mix that the State Health Department's clinic actually faces day to day.

> **A question that often comes up here:** *"Where did the validation set go? Earlier notebooks split 60/20/20."* It was a teaching step. In nb01–nb07 the validation set gave you one untouched sample for checking a model before you had cross-validation; since nb08, 5-fold CV on the training set does that job better, because every training row takes a turn as held-out data and you get five scores instead of one. Keeping a validation slice would only take rows away from the training set and add a second, noisier opinion. The test set is different — it stays sealed until nb14's selection ceremony. Treat `X_test_clf`, `y_test_clf`, `X_test_reg`, and `y_test_reg` like sealed envelopes in the State Health Department's audit room: they exist, they have your name on them, and you open them exactly once.

**Key takeaway:** Two cases, two namespaces, one 80/20 split each, two sealed envelopes. Everything from here through Section 7 evaluates on the **training** sets via 5-fold CV, and `X_test_*` stays closed until nb14.

---


## 4. Classification Tree Example — Wisconsin Breast Cancer

Time to run the algorithm on the real classification problem. The fit below trains a `DecisionTreeClassifier` with `max_depth=3` on the breast-cancer training set — three layers of yes/no questions, no more. That depth is shallow on purpose. Three splits is plenty for a strong first baseline on this dataset, and it is small enough that `plot_tree` produces a flowchart a clinician can actually read on a single page.

Three numbers tell you how the fit went: the **training accuracy** (how well the tree memorizes the 455 patients it saw during fitting), the **5-fold CV accuracy** (how well the same algorithm generalizes when trained and scored on different folds), and the **gap** between them. A large gap is the early-warning siren for overfitting. We will also visualize the tree itself, because the headline value of trees in a regulated setting is not just *"the model got X% accuracy"* — it is *"here is the four-question rule the model used, and a pathologist can sanity-check every split against the clinical literature."*

> 💡 **Gemini Prompt:** "Fit a DecisionTreeClassifier(max_depth=3, random_state=474) on X_train_clf, y_train_clf. Print training accuracy on the full training set and 5-fold CV accuracy mean ± SD using cv_clf and scoring='accuracy'. Then call plot_tree with feature_names, class_names, filled=True."
>
> **After running, verify:**
> - [ ] Train accuracy in the 0.96–0.99 range
> - [ ] CV mean ± SD reported with 4-digit precision
> - [ ] Train-CV gap printed explicitly
> - [ ] Tree visualization shows ≤ 8 leaves


In [ ]:
# Classification tree at max_depth=3 — fit, score, visualize.
tree_clf = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED)
tree_clf.fit(X_train_clf, y_train_clf)

train_acc_clf = tree_clf.score(X_train_clf, y_train_clf)
cv_scores_clf = cross_val_score(tree_clf, X_train_clf, y_train_clf,
                                cv=cv_clf, scoring='accuracy')
cv_mean_clf = cv_scores_clf.mean()
cv_std_clf  = cv_scores_clf.std(ddof=1)

print("=== CLASSIFICATION TREE (max_depth=3) ===")
print(f"Train accuracy:        {train_acc_clf:.4f}")
print(f"5-fold CV accuracy:    {cv_mean_clf:.4f}  (SD = {cv_std_clf:.4f})")
print(f"Overfit gap (train−CV): {train_acc_clf - cv_mean_clf:.4f}")
print(f"Tree leaves:           {tree_clf.get_n_leaves()}")

# --- Visualize the tree ---
fig, ax = plt.subplots(figsize=(20, 9))
plot_tree(
    tree_clf,
    feature_names=X_clf.columns,
    class_names=data_clf.target_names,
    filled=True,
    rounded=True,
    fontsize=10,
    ax=ax,
)
ax.set_title('Wisconsin Breast Cancer — Classification Tree at max_depth=3',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Each leaf shows: gini impurity, sample count, class distribution, predicted class.")
print("💡 Darker shading = purer leaf (mostly one class).")


**Reading the output:**

The `max_depth=3` tree on the breast-cancer training set reaches a training accuracy of **0.978** and a 5-fold CV accuracy of **0.930** (fold SD 0.024), a train–CV gap of about **0.048**. The gap says the tree fits its own training rows noticeably better than rows it has not seen — some memorization, even at three questions per path. The tree itself uses **eight leaves**; you can read every decision path in plain English, and that is exactly what makes it deployable in a regulated clinical workflow. The root split lands on `worst radius` (at 16.8), a well-established malignancy marker in the clinical literature, and one path — worst radius above 16.8, worst concavity above 0.22, texture error above 0.45 — holds 139 training biopsies, every one of them malignant. That is the kind of plausibility check that builds trust with the State Health Department's review board when they audit the model.

> **A question that often comes up here:** *"What does the gini number on each node mean?"* Gini impurity is `1 − Σ p_i²` where `p_i` is the fraction of samples in class `i`. A pure leaf (all malignant or all benign) has gini = 0; a perfectly mixed 50/50 leaf has gini = 0.5. The tree's split criterion at every internal node is the same simple rule: *pick the feature and threshold that drop the weighted-average gini of the two children the most*. That single objective — minimize child impurity — is the entirety of how a classification tree decides where to cut.

**Key takeaway:** A shallow classification tree is interpretable and clinically plausible, but accuracy and the train–CV gap are diagnostics, not a comparison with anything. Section 6 sweeps depth on both cases to make the bias-variance trade-off visible, Section 7 puts the tree against the logistic reference on ROC-AUC, and Exercise 1 nominates a depth with the one-standard-error heuristic and tests the nominee against depth 3.

---


## 5. Regression Tree Example — California Housing

The regression tree on California Housing uses the same machinery you just saw on the breast-cancer data — recursive binary splits, axis-aligned cuts, a constant prediction per leaf — with two natural adaptations for a continuous target. First, the **split criterion** switches from gini impurity to **mean squared error**: the tree now picks the feature and threshold that drop the weighted-average MSE of the two children the most. Same logic, different metric. Second, the **leaf prediction** is no longer a majority-class vote; it is the **average house value** of all training tracts that landed in that leaf. A leaf containing 47 tracts whose median values average USD 240K predicts USD 240K for any new property that follows the same path through the tree.

The visualization below packs three diagnostics into a single figure. The top panel shows the fitted tree itself, the same way you saw the classification tree in the previous section. The bottom-left panel is the **predicted-vs-actual scatter** — the mandatory diagnostic for any regression model. The bottom-right panel slides `MedInc` (median income) across its full training range while holding every other feature at its training median; that traces a one-dimensional cross-section through the tree's prediction surface and produces the **step function** that is the visual signature of every regression tree.

> 💡 **Gemini Prompt:** "Fit DecisionTreeRegressor(max_depth=3, random_state=474) on X_train_reg, y_train_reg. Print train R², 5-fold CV R² mean ± SD using cv_reg, and train RMSE in dollars. Plot three panels: plot_tree, predicted-vs-actual scatter (use the helper), and a step-function showing tree.predict over MedInc (with other features held at their training median). Use cross_val_score with scoring='neg_root_mean_squared_error' for the RMSE CV."
>
> **After running, verify:**
> - [ ] Train R² in 0.50–0.70 range; CV R² slightly lower
> - [ ] CV RMSE reported in USD (multiply by 100,000)
> - [ ] Step-function plot shows 8 distinct horizontal levels (one per leaf)
> - [ ] Predicted-vs-actual scatter has the y=x reference line drawn


In [ ]:
# Regression tree at max_depth=3 — fit, score, visualize, step-function.
tree_reg = DecisionTreeRegressor(max_depth=3, random_state=RANDOM_SEED)
tree_reg.fit(X_train_reg, y_train_reg)

train_r2_reg  = tree_reg.score(X_train_reg, y_train_reg)
cv_r2_reg     = cross_val_score(tree_reg, X_train_reg, y_train_reg,
                                cv=cv_reg, scoring='r2')
cv_rmse_reg   = -cross_val_score(tree_reg, X_train_reg, y_train_reg,
                                 cv=cv_reg, scoring='neg_root_mean_squared_error')

print("=== REGRESSION TREE (max_depth=3) ===")
print(f"Train R²:               {train_r2_reg:.4f}")
print(f"5-fold CV R²:           {cv_r2_reg.mean():.4f}  (SD = {cv_r2_reg.std(ddof=1):.4f})")
print(f"5-fold CV RMSE:         {cv_rmse_reg.mean():.4f} (in 100K USD units)")
print(f"5-fold CV RMSE in USD:  USD {cv_rmse_reg.mean()*100_000:,.0f}")
print(f"Tree leaves:            {tree_reg.get_n_leaves()}")

# --- Three-panel diagnostic ---
fig = plt.figure(figsize=(20, 11))
gs  = fig.add_gridspec(2, 2, height_ratios=[1.4, 1])

# Top: the regression tree itself
ax_tree = fig.add_subplot(gs[0, :])
plot_tree(
    tree_reg,
    feature_names=X_reg.columns,
    filled=True, rounded=True, fontsize=10, ax=ax_tree,
)
ax_tree.set_title('California Housing — Regression Tree at max_depth=3',
                  fontsize=14, fontweight='bold')

# Bottom-left: predicted-vs-actual on training set
ax_pva = fig.add_subplot(gs[1, 0])
y_pred_reg = tree_reg.predict(X_train_reg)
plot_predicted_vs_actual(y_train_reg, y_pred_reg, ax_pva,
                         title='Predicted vs Actual (training set, USD 100K units)')

# Bottom-right: 1D step function — predict as MedInc varies, others at median
ax_step = fig.add_subplot(gs[1, 1])
medinc_range = np.linspace(X_train_reg['MedInc'].min(),
                           X_train_reg['MedInc'].max(), 500)
median_row = X_train_reg.median()
grid = pd.DataFrame([median_row] * 500, columns=X_train_reg.columns)
grid['MedInc'] = medinc_range
y_step = tree_reg.predict(grid)

ax_step.scatter(X_train_reg['MedInc'], y_train_reg,
                alpha=0.10, s=6, color=GREY, label='Training data')
ax_step.plot(medinc_range, y_step, color=REG_COLOR, linewidth=2.5,
             label='Tree prediction (other features at median)')
ax_step.set_xlabel('MedInc (median income, 10K USD units)')
ax_step.set_ylabel('Predicted house value (USD 100K units)')
ax_step.set_title('Tree predicts step-functions, not smooth curves',
                  fontsize=12, fontweight='bold')
ax_step.legend()
ax_step.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 The step function has exactly as many levels as the tree has leaves.")
print("💡 This is the visual signature of axis-aligned splits — no smoothness.")


**Reading the output:**

The `max_depth=3` regression tree on California Housing lands a 5-fold CV R² of **0.525** (fold SD 0.014) and a CV RMSE of about **USD 79,400**. In plain terms, three questions per path already explain about half of the variation in house values that a *"predict the training mean"* rule leaves unexplained — the rule HomeValue Analytics' pricing analysts had been applying by hand before this engagement. Whether any model beats the linear reference by an amount the CFO would act on is a separate question, and Section 7 answers it with a paired comparison rather than a promise.

The eight leaves at the bottom of the top panel are the **entire prediction repertoire** of this tree. Every leaf reports a `value` line — the **mean house value** of every training tract that followed that leaf's specific path of yes/no answers, and exactly the price the tree quotes for any new property that traces the same path. The eight leaf-means span roughly **USD 116K** at the cheapest end (`MedInc` at most 3.07 with more than 4.34 rooms per household) to **USD 468K** at the most expensive (`MedInc` above 8.36 — the wealthiest tracts in training). The root split is on income: tracts with `MedInc ≤ 5.04` (median household income up to about USD 50K) land in four leaves between USD 116K and USD 277K; tracts with `MedInc > 5.04` land in four leaves between USD 267K and USD 468K. A `max_depth=3` tree can quote exactly **eight** prices for any property you hand it — and the step-function panel on the bottom-right shows those leaf-means along the `MedInc` axis with everything else held at its training median, which is why it has only a handful of horizontal levels.

The step-function plot in the bottom-right panel is the visual signature of every regression tree. Across the entire range of `MedInc`, the prediction takes only a few distinct values — never more than the tree has leaves. Inside each step the tree is constant; at the step boundaries it jumps. This has one important consequence: a regression tree **cannot extrapolate**. If a new property has a `MedInc` higher than anything seen in training, the tree clamps to the highest-leaf average — it has no notion of *"the price keeps rising past USD 500K."* Linear models extrapolate cleanly along a regression line; trees do not. The trade you are making is interpretability and non-linearity in exchange for boundedness and step-function predictions, and that trade is the central characteristic of the algorithm.

> **A question that often comes up here:** *"If the tree only predicts step values, isn't that always worse than a linear model?"* Not always. When the true relationship is **non-monotonic** — for example, when neighbourhood effects cause house prices to drop in certain income brackets — the tree captures that bend, and a linear model cannot. When the relationship is **monotonic and smooth**, the linear model has the advantage. Section 7 makes that comparison concrete on both cases against the nb09 reference pipelines.

**Key takeaway:** Regression trees buy you interpretability and non-linearity at the cost of step-function predictions and zero extrapolation. Whether that trade is worth making depends on the data — which is exactly what the next two sections quantify on the two real cases.

---


## 6. The Overfitting Problem — Paired Across Both Cases

Sections 4 and 5 each fit a single tree at `max_depth=3`. The obvious next question is: *what would happen at depth 1, or depth 5, or with no depth cap at all?* The cell below answers that on **both cases at the same time** by sweeping `max_depth ∈ {1, 2, 3, 5, 10, 20, None}` and recording the training score and the 5-fold CV score at every setting. The result is a two-panel plot — classification on the left, regression on the right — that puts the bias-variance trade-off in front of you in a single picture.

Read two numbers at every depth. The **CV score** is the held-out evidence: how well trees of that depth predict rows they were not fitted on. The **gap** between the training score and the CV score tells you how much the tree memorizes its own training rows. A shallow tree under-asks: training and CV are both mediocre and the gap is small (high bias). A deep tree memorizes: training climbs toward 1.0 and the gap grows (high variance). The gap is a diagnostic, not a verdict — a deeper tree with a bigger gap can still score higher on the held-out folds, and when it does, that held-out score is real. The CV curve tells you where accuracy peaks; the stakeholder's readability condition from the Why This Matters cell tells you how deep a single tree is allowed to be.


> 💡 **Gemini Prompt:** "Sweep max_depth in [1, 2, 3, 5, 10, 20, None] for both DecisionTreeClassifier on X_train_clf and DecisionTreeRegressor on X_train_reg. For each depth and each case, fit on the full training set, record the training score and the 5-fold CV mean and SD (cv_clf with accuracy, cv_reg with R²). Build a 1×2 subplot using the plot_train_cv_curve helper — left panel classification (accuracy), right panel regression (R²) — and print a table of train score, CV score and their gap by depth."
>
> **After running, verify:**
> - [ ] Both panels share the same x-axis layout (same depth values)
> - [ ] Classification CV accuracy stays close to 0.93 from depth 3 onward
> - [ ] Regression CV R² peaks at an intermediate depth and falls after it
> - [ ] Train curves both reach 1.0 at unrestricted depth (None)


In [ ]:
# Paired depth sweep — same pattern emerges on both cases.
depths = [1, 2, 3, 5, 10, 20, None]

# --- Classification sweep ---
clf_train, clf_cv_mean, clf_cv_sd = [], [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_SEED).fit(X_train_clf, y_train_clf)
    clf_train.append(t.score(X_train_clf, y_train_clf))
    s = cross_val_score(t, X_train_clf, y_train_clf, cv=cv_clf, scoring='accuracy')
    clf_cv_mean.append(s.mean())
    clf_cv_sd.append(s.std(ddof=1))

# --- Regression sweep ---
reg_train, reg_cv_mean, reg_cv_sd = [], [], []
for d in depths:
    t = DecisionTreeRegressor(max_depth=d, random_state=RANDOM_SEED).fit(X_train_reg, y_train_reg)
    reg_train.append(t.score(X_train_reg, y_train_reg))
    s = cross_val_score(t, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2')
    reg_cv_mean.append(s.mean())
    reg_cv_sd.append(s.std(ddof=1))

# --- Side-by-side panels ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_train_cv_curve(
    [str(d) for d in depths], clf_train, clf_cv_mean, clf_cv_sd,
    xlabel='max_depth', ylabel='Accuracy',
    title='Classification — Wisconsin Breast Cancer',
    ax=axes[0], color_cv=CLF_COLOR
)
plot_train_cv_curve(
    [str(d) for d in depths], reg_train, reg_cv_mean, reg_cv_sd,
    xlabel='max_depth', ylabel='R²',
    title='Regression — California Housing',
    ax=axes[1], color_cv=REG_COLOR
)

fig.suptitle('Same overfitting pattern on both spines — train rises to 1.0; CV plateaus and then degrades',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Print the train–CV gap table
print("=== TRAIN − CV GAP BY DEPTH ===")
print(f"{'depth':>8} | {'clf train':>10} | {'clf CV':>10} | {'gap':>8} ||"
      f" {'reg train':>10} | {'reg CV':>10} | {'gap':>8}")
for i, d in enumerate(depths):
    print(f"{str(d):>8} | {clf_train[i]:>10.4f} | {clf_cv_mean[i]:>10.4f} | "
          f"{clf_train[i]-clf_cv_mean[i]:>8.4f} || "
          f"{reg_train[i]:>10.4f} | {reg_cv_mean[i]:>10.4f} | "
          f"{reg_train[i]-reg_cv_mean[i]:>8.4f}")


**Reading the output:**

The two panels tell the same story at different scales. On the left, classification accuracy rises from **0.910** at depth 1 to **0.930** at depth 3 and then barely moves: **0.932** at depth 5, **0.928** from depth 10 on. On the right, regression R² climbs from **0.307** at depth 1 to **0.611** at depth 5 and a peak of **0.681** at depth 10, then falls back to **0.609** at depth 20. In **both** cases the training score keeps marching toward 1.0 — the classification tree is perfect on its own training rows from depth 10, the regression tree from no cap at all — and the train–CV gap grows with depth: on classification from 0.015 at depth 1 to 0.073 at depth 10, on regression from 0.007 to 0.155 at depth 10 and 0.39 with no cap.

Notice where the CV curve peaks on each side. Classification flattens almost immediately because the dataset is small — 455 training patients, so after three questions the leaves are already small and extra depth mostly memorizes. Regression keeps gaining until depth 10 because there are over 16,000 training tracts: a depth-10 tree still has hundreds of rows behind most of its leaves. Bigger training sets support deeper trees.

Now the depth this notebook carries forward. For **MedScreen**, depth 3 meets the review board's three-question condition, and the sweep shows no visible cost: depth 5 scores 0.932 against depth 3's 0.930, a difference of 0.002 accuracy — tiny next to the fold-to-fold spread (SD 0.024 at depth 3). For **HomeValue**, the legal team's five-question condition caps the single tree at depth 5 (CV R² 0.611, gap 0.027). Be honest about what the cap costs: depth 10 scores 0.07 R² higher on the held-out folds. That is not overfitting credit — CV scores are measured on rows the trees never saw, so the gap at depth 10 means the tree memorizes a lot, *and* the CV score says it still predicts unseen tracts better than depth 5. What the gap cannot tell you is whether that gain is worth an unreadable tree; that is the stakeholder's call, declared before scoring. Past depth 10 the CV score does fall — that is where memorization finally costs held-out accuracy. Exercise 2 puts the depth-10 gain through `compare_paired`.

> **A question that often comes up here:** *"Why does CV sometimes degrade after the peak and sometimes stay flat?"* When CV degrades, the deep tree has carved out tiny leaves that fit noise specific to the training rows, and on the held-out folds those leaves predict the wrong class or the wrong leaf-mean — the regression panel after depth 10. When CV stays flat, the tree is still memorizing, but the noisy leaves cost about as much as the extra detail gains, so the average does not move — the classification panel after depth 3. Either way the **train–CV gap** grows with depth, which makes it a good warning light for memorization; the CV score is still the number that says how well the tree predicts new rows.

**Key takeaway:** The overfitting pattern is universal — same shape, different scale. Train climbs toward 1.0 and the gap grows; CV rises, peaks, then flattens or falls. This notebook carries **depth 3** for MedScreen (the review board's three-question condition, at no visible cost) and **depth 5** for HomeValue (the legal team's five-question condition, which costs about 0.07 R² against depth 10 on the held-out folds). Exercises 1 and 2 nominate depths on a finer grid with the one-standard-error heuristic and test each nominee against the carried depth; Section 7 asks whether either carried tree has earned the right to displace the nb09 linear reference.

---


## 7. Tree vs nb09 Reference — Paired

Time for the head-to-head. Each tree is the **challenger** (A) and the **nb09 reference model** for its case is the **current champion** (B) — the linear baseline that survived nb09's tuned-vs-default comparison:

- **Classification:** `DecisionTreeClassifier(max_depth=3)` vs `reference_clf` = `Pipeline([StandardScaler, LogisticRegression(C=1.0, max_iter=5000)])`. Primary metric: ROC-AUC.
- **Regression:** `DecisionTreeRegressor(max_depth=5)` vs `reference_reg` = `Pipeline([StandardScaler, LinearRegression()])` (OLS). Primary metric: R².

The output has two parts. The **description**: each model's five fold scores, their mean and their SD, printed as a table and drawn as a dot strip (the bar marks the mean). There are no error bars on a single model's score — the Q&A below explains why. The **decision**: `compare_paired(tree_scores, reference_scores, margin)` on the identical folds (same splitter object, so fold *i* holds the same rows for both models). It subtracts fold by fold, `d_i = score_tree(i) − score_reference(i)`, which removes the fold-to-fold difficulty the two models share — a hard fold is hard for both. Then it builds an interval around the mean gain with a **corrected** standard error, `t × SD(d) × √(1/k + 1/(k−1))` (Nadeau and Bengio, 2003): the extra `1/(k−1)` term widens the interval because the five training sets overlap, so the five gains are not independent.

The corrected interval is read against the base margin δ = 0.01 declared in the Why This Matters cell, and exactly one of four outcomes applies:

1. **Lower end above +δ** → *A is better by a worthwhile amount*: the tree replaces the reference.
2. **Upper end below −δ** → *B is better by a worthwhile amount*: the reference stays, and the evidence says it is clearly better.
3. **Whole interval inside (−δ, +δ)** → *practically equivalent*: choose the simpler model.
4. **Anything else** → *unresolved*: the evidence cannot tell; apply the default you declared — the current champion stays.

Only outcome 1 changes the champion. How many folds the tree wins is description, never the verdict: a challenger can win all five folds and still be unresolved. `show_comparison` also re-reads the same interval at the small and large margins and reports whether the verdict is **robust** across the band or **margin-sensitive**. One distinction worth keeping straight: the one-standard-error heuristic you will use in the exercises *nominates* a setting within one family's tuning curve; `compare_paired` is how a nominee, or any other challenger, earns a place against the current champion.

> **A question that often comes up here:** *"Why don't we put a 95% confidence interval around each model's CV score, the way a statistics course would?"* Because the arithmetic `mean ± t·s/√k` assumes the five fold scores are independent, and they are not: any two folds share three-quarters of their training rows, so the scores move together and the interval comes out too narrow — Bengio and Grandvalet (2004) showed there is no unbiased way to estimate that variability from one run of k-fold CV, and Bates, Hastie and Tibshirani (2024) found such intervals cover the truth far less often than 95%. Worse, the question the HomeValue CFO actually asks is never "how good is Ridge?" in isolation but "is the forest better than what we run today by enough to matter?", and two separate intervals cannot answer that: they can overlap generously while one model wins on every fold. So you report each model's fold scores, mean and spread, and you put the interval where it belongs — around the per-fold *gain* of one model over the other, corrected for the shared training rows, read against the margin you declared.


> 💡 **Gemini Prompt:** "Using the compare_paired function, DELTAS_AUC and DELTAS_R2 from the setup cell, compare DecisionTreeClassifier(max_depth=3) against reference_clf (the nb09 LogReg pipeline) on X_train_clf with cv_clf and scoring='roc_auc', and DecisionTreeRegressor(max_depth=5) against reference_reg (the nb09 OLS pipeline) on X_train_reg with cv_reg and scoring='r2'. Pass the same cv object to both cross_val_score calls on a case and keep each model's five fold scores. Print a table per case with the five fold scores, mean and SD of each model — no confidence interval for a single model. Then call compare_paired(tree_scores, reference_scores, margin) with the tree as A (challenger) and the reference as B (current champion), print the per-fold gains, the mean gain, the corrected interval and the verdict at the base margin, and repeat the verdict at the small and large margins. Plot each case's fold scores as a dot strip, and each case's per-fold gains with the corrected interval drawn against a shaded ±δ band."
>
> **After running, verify:**
> - [ ] Both models on a case use the same `cv_clf` / `cv_reg` object, so the folds are identical
> - [ ] Tables show five fold scores, `mean` and `sd` — no interval around a single model
> - [ ] For each case: per-fold gains, mean gain, corrected interval, and one of the four verdicts at δ = 0.01
> - [ ] The band check reports the verdict at δ = 0.005, 0.01 and 0.02
> - [ ] The decision panels show the interval against the shaded ±δ band and a line at zero


In [ ]:
# Tree vs nb09 reference — identical folds; each model's fold scores described, the per-fold gain decides.
tree_clf_d3 = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED)
tree_reg_d5 = DecisionTreeRegressor(max_depth=5, random_state=RANDOM_SEED)

# One splitter object per case (cv_clf, cv_reg), passed to BOTH cross_val_score calls → identical folds.
scores_clf = {
    'Decision Tree (depth=3)':       cross_val_score(tree_clf_d3, X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc'),
    'nb09 reference: LogReg(C=1.0)': cross_val_score(reference_clf, X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc'),
}
scores_reg = {
    'Decision Tree (depth=5)':       cross_val_score(tree_reg_d5, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2'),
    'nb09 reference: OLS':           cross_val_score(reference_reg, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2'),
}

print("=== CLASSIFICATION — 5-fold CV ROC-AUC: fold scores, mean, SD ===")
print(fold_table(scores_clf).to_string(index=False))
print()
print("=== REGRESSION — 5-fold CV R²: fold scores, mean, SD ===")
print(fold_table(scores_reg).to_string(index=False))

# --- The decision: challenger (tree) = A, current champion (nb09 reference) = B ---
print()
d_clf, gain_clf, ci_clf, verdict_clf = show_comparison(
    'Decision Tree (depth=3)', 'LogReg(C=1.0)',
    scores_clf['Decision Tree (depth=3)'], scores_clf['nb09 reference: LogReg(C=1.0)'],
    DELTAS_AUC, 'ROC-AUC')
print()
d_reg, gain_reg, ci_reg, verdict_reg = show_comparison(
    'Decision Tree (depth=5)', 'OLS',
    scores_reg['Decision Tree (depth=5)'], scores_reg['nb09 reference: OLS'],
    DELTAS_R2, 'R²')

champion_clf = 'Decision Tree (depth=3)' if verdict_clf.startswith('A is better') else 'LogReg(C=1.0)'
champion_reg = 'Decision Tree (depth=5)' if verdict_reg.startswith('A is better') else 'OLS'
print()
print(f"MedScreen champion after this comparison: {champion_clf}")
print(f"HomeValue champion after this comparison: {champion_reg}")

# --- Description (top row) and decision (bottom row) ---
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
plot_fold_scores(axes[0, 0], scores_clf, '5-fold CV ROC-AUC', 'MedScreen — fold scores (bar = mean)',
                 colors={'Decision Tree (depth=3)': CLF_COLOR})
plot_fold_scores(axes[0, 1], scores_reg, '5-fold CV R²', 'HomeValue — fold scores (bar = mean)',
                 colors={'Decision Tree (depth=5)': REG_COLOR})
plot_gain_intervals(axes[1, 0], [('tree − LogReg', d_clf, ci_clf)], DELTA_AUC,
                    'Gain in ROC-AUC (tree − LogReg)', 'MedScreen — the decision: corrected interval vs ±δ')
plot_gain_intervals(axes[1, 1], [('tree − OLS', d_reg, ci_reg)], DELTA_R2,
                    'Gain in R² (tree − OLS)', 'HomeValue — the decision: corrected interval vs ±δ')
fig.suptitle("Tree vs nb09 reference — fold scores describe each model; the paired gain's interval decides",
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**Reading the output:**

Start with the description. On the breast-cancer side, the **classification tree at depth 3** scores a mean 5-fold CV ROC-AUC of **0.940**, but its fold scores spread from **0.859 to 0.993** (SD 0.051) — on only about 91 held-out patients per fold, one unlucky fold moves the score a lot. The nb09 reference (`StandardScaler + LogisticRegression(C=1.0)`) scores **0.994** on average, between **0.982 and 0.9995** (SD 0.007) — two of its folds are already at 0.9995, so on those folds no challenger could gain more than about 0.0005. On the California-housing side, the **regression tree at depth 5** averages a CV R² of **0.611** (SD 0.015) and OLS **0.604** (SD 0.019).

Now the decision. **MedScreen:** the tree's per-fold gains over LogReg are −0.133, −0.034, −0.006, −0.039 and −0.059 — it trails on all five folds, by a mean of **−0.054** ROC-AUC. Yet the corrected interval runs from **−0.143 to +0.035**: the one very bad fold makes the gains so uneven that the evidence cannot even rule out a small gain for the tree. The interval crosses both −0.01 and +0.01, so the outcome is ***unresolved*** — at the small, base and large margins alike (robust across the band). Unresolved means the declared default applies: **the review board keeps `LogisticRegression(C=1.0)`**. Notice what did *not* decide this: five losing folds out of five. Fold signs describe; the interval decides.

**HomeValue:** the depth-5 tree's per-fold gains over OLS are +0.010, +0.013, −0.008, +0.014 and +0.003, a mean gain of **+0.006** R², with a corrected interval of **[−0.011, +0.024]**. The interval reaches above the 0.01 margin, so a worthwhile gain is not ruled out, but it also reaches below zero, so the evidence cannot show one: ***unresolved***, again at every margin in the band. **HomeValue keeps OLS**. In words the CFO can use: *"The evidence does not show a gain large enough to justify the tree, so under our policy we keep OLS."* The tree is not shown to be worse, and it is not shown to be equivalent; the five folds simply cannot tell.

> **A question that often comes up here:** *"On classification the nb09 LogReg still wins — why bother with trees at all?"* Three answers. First, today's single tree is the building block for the next two notebooks: Random Forests (nb12) and Gradient Boosting (nb13) are ensembles **of trees**, and they will face the same reference under the same rule — with no promise about how that turns out. Second, trees give you a kind of **interpretability** the linear baseline simply cannot match — a State Health Department clinician can read three yes/no questions off a flowchart, but no one is reading 30 standardized logistic coefficients in their head. Third, the comparison itself is the discipline: every new model is scored on the same folds and judged against the same margin, which is exactly what lets nb14's selection ceremony name a defensible champion for each case.

**Key takeaway:** Two stakeholders, one verdict each, both decided by the corrected interval of the per-fold gain against δ = 0.01. **MedScreen keeps `LogisticRegression(C=1.0)`** — the tree vs LogReg comparison is unresolved (mean gain −0.054, interval [−0.143, +0.035]). **HomeValue keeps OLS** — the depth-5 tree vs OLS comparison is unresolved (mean gain +0.006, interval [−0.011, +0.024]). Both verdicts are robust across the band. The single tree carries through the rest of the course in two ways: as the readable tree-family candidate in nb14's roster, and as the building block of the ensembles in nb12 and nb13.

---


## 📝 PAUSE-AND-DO Exercise 1 (clf, 5 minutes) — Nominate and Test a Classification Tree Depth

**Task:** Use 5-fold cross-validation to nominate a `max_depth` for `DecisionTreeClassifier` on the Wisconsin breast cancer training set, scoring on **ROC-AUC**, and then test your nominee against depth 3, the depth this notebook carries forward.

**Instructions:**
1. Sweep `max_depth ∈ [2, 3, 4, 5, 6, 7, 8, 10, 15]`, keeping each depth's five fold scores (all from `cv_clf`).
2. For each depth, compute the 5-fold CV mean and SD.
3. Plot CV mean with SD error bars across depths.
4. Nominate a depth with the **one-standard-error heuristic**: find the depth with the highest CV mean, compute its standard error as SD/√5, and nominate the simplest depth whose CV mean is at least *best mean − one SE*.
5. Compare your nominee (A) against depth 3 (B) with `compare_paired(nominee_scores, depth3_scores, DELTA_AUC)` and state which depth you carry forward.
6. Write 3 short findings: what does the curve tell you about the bias-variance trade-off, and why is a nominee not yet a verdict?

---


> 💡 **Gemini Prompt:** "Cross-validate DecisionTreeClassifier with random_state=474 over depths [2,3,4,5,6,7,8,10,15] on X_train_clf, y_train_clf using cv_clf and scoring='roc_auc', and keep each depth's five fold scores in a dictionary. Report the mean and SD for each depth in a DataFrame and plot the mean with SD error bars. Apply the one-standard-error heuristic: take the depth with the highest mean, compute SE = SD/sqrt(5) for that depth, and nominate the simplest depth whose mean is at least best mean − SE. Then run compare_paired(nominee fold scores, depth-3 fold scores, DELTA_AUC) and print the per-fold gains, the corrected interval and the verdict."
>
> **After running, verify:**
> - [ ] DataFrame has 9 rows (one per depth)
> - [ ] Plot uses error bars from the SD column
> - [ ] Best depth, the one-SE threshold (best mean − SD/√5) and the nominee all printed
> - [ ] `compare_paired` output shows per-fold gains, a corrected interval and one of the four verdicts
> - [ ] CV ROC-AUC values are in the 0.92–0.95 range


In [ ]:
# YOUR SOLUTION CODE HERE
# Nominate a classification-tree depth with 5-fold CV ROC-AUC and the one-SE heuristic (SD/√5),
# then compare the nominee against depth 3 with compare_paired on the same folds.


## 📝 PAUSE-AND-DO Exercise 2 (reg, 5 minutes) — Nominate and Test a Regression Tree Depth

**Task:** Use 5-fold cross-validation to nominate a `max_depth` for `DecisionTreeRegressor` on the California housing training set, scoring on **R²**, and then test your nominee against depth 5, the depth this notebook carries forward.

**Instructions:**
1. Sweep `max_depth ∈ [2, 3, 5, 7, 10, 15, 20]`, keeping each depth's five fold R² scores (all from `cv_reg`).
2. For each depth, compute the 5-fold CV mean and SD of R², the mean CV RMSE in USD, and the number of leaves of a tree fit on all of `X_train_reg`.
3. Plot CV mean with SD error bars across depths.
4. Apply the same one-standard-error heuristic (SD/√5) to nominate the simplest competitive depth.
5. Compare your nominee (A) against depth 5 (B) with `compare_paired(nominee_scores, depth5_scores, DELTA_R2)`.
6. Write 3 short findings: same shape as Exercise 1, different scale? And what should HomeValue do with the verdict, given the legal team's five-question condition?

---


> 💡 **Gemini Prompt:** "Cross-validate DecisionTreeRegressor with random_state=474 over depths [2,3,5,7,10,15,20] on X_train_reg, y_train_reg using cv_reg, scoring both 'r2' and 'neg_root_mean_squared_error', and keep each depth's five fold R² scores. For each depth also fit the tree on all of X_train_reg and record get_n_leaves(). Report a DataFrame with the R² mean and SD, the RMSE in USD (multiply by 100,000) and the leaf count, and plot the R² mean with SD error bars. Nominate a depth with the one-standard-error heuristic (SE = SD/sqrt(5) of the best depth). Then run compare_paired(nominee fold scores, depth-5 fold scores, DELTA_R2) and print the per-fold gains, the corrected interval and the verdict."
>
> **After running, verify:**
> - [ ] DataFrame has 7 rows (one per depth), with an RMSE column in USD and a leaf-count column
> - [ ] Plot uses error bars from the SD column
> - [ ] Best depth, the one-SE threshold and the nominee all printed
> - [ ] `compare_paired` output shows per-fold gains, a corrected interval and one of the four verdicts
> - [ ] CV R² values are in the 0.44–0.69 range


In [ ]:
# YOUR SOLUTION CODE HERE
# Nominate a regression-tree depth with 5-fold CV R² and the one-SE heuristic (SD/√5),
# report RMSE in USD and leaf counts, then compare the nominee against depth 5 with compare_paired.


## 8. Wrap-Up — Key Takeaways

**What landed today:**

1. **Decision trees partition feature space with axis-aligned splits.** Every leaf holds a constant prediction — majority class for classification, leaf-mean for regression. Tree depth controls how finely the space is partitioned.
2. **The overfitting pattern is universal, and the gap is a diagnostic, not a verdict.** On both cases the training score climbs toward 1.0 while the CV score rises, peaks and flattens or falls. The train–CV gap warns you about memorization; the CV score is the held-out evidence. The one-standard-error heuristic (SD/√5) nominates a depth; a nominee then has to face the depth you carry forward on the same folds.
3. **A challenger has to earn a worthwhile gain.** `compare_paired` scores both models on identical folds, subtracts fold by fold, builds an interval corrected for the overlap between folds, and reads it against the base margin: worthwhile gain, worthwhile loss, practically equivalent, or unresolved. Today both trees came out *unresolved* against their nb09 references — MedScreen's even though it trailed on every fold, HomeValue's even though it led on four — so **LogReg(C=1.0)** and **OLS** stay. Fold wins describe; the interval decides.
4. **Declared conditions are part of the decision.** The review board's three-question flowchart and the legal team's five-question limit were stated before any score was computed. Exercise 2 found a deeper regression tree that is better by a worthwhile amount, and the readable tree still stays at depth 5, because that tree is not eligible as a single-tree deliverable.
5. **Two parallel cases, two namespaces, two locked test sets.** From here through nb14, every section of every notebook pairs the classification and regression tracks under the `_clf` / `_reg` suffix convention, on one 80/20 split per case with 5-fold CV on the training rows.

**Bridge to nb12 — Random Forests:**

A single tree is high-variance: a small change in the training data can flip the root split. Random forests fix that by **averaging many deep trees**, each trained on a bootstrap sample with a random feature subset. The same dual-case structure continues — `RandomForestClassifier` on Wisconsin breast cancer, `RandomForestRegressor` on California housing — and so does the rule: the forest is a challenger, the nb09 reference is the current champion, and the forest replaces it only if the corrected interval of its per-fold gain clears **0.01** (R² for HomeValue, ROC-AUC for MedScreen). Exercise 2 already showed where extra accuracy hides on HomeValue — in deeper trees that stop being readable — and a forest is built to collect it. nb12 also tunes the forest with a grid search, and there the grid's best cell is only a **nominee**: it has to beat the default forest on the same folds before it replaces it. Finally, nb12 introduces the **four-method feature-importance reconciliation table** that nb13 and nb14 lean on for interpretation — and that you will reuse when you write your final-project interpretation section.

> **A question that often comes up at this point:** *"If forests beat trees on every benchmark, why teach single trees first?"* Two reasons. First, the forest is an average of trees — without understanding what one tree does, the averaging operation is opaque. Second, single trees are still the right answer when the deliverable demands a single auditable decision path (clinical screening, regulatory compliance, courtroom defense). Forests trade interpretability for stability; that trade is sometimes worth making and sometimes not, and you cannot evaluate the trade without first having seen what gets lost.

---


## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete both PAUSE-AND-DO exercises** — Exercise 1 (classification depth tuning) and Exercise 2 (regression depth tuning).
2. **Run All Cells** — `Runtime → Run all` to ensure every cell executes without error.
3. **Save a Copy** — `File → Save a copy in Drive`, or download as `.ipynb`.
4. **Submit** — upload the `.ipynb` file to the Notebook 11 participation assignment on Brightspace.

### Before Submitting, Check:

- [ ] Both exercise solutions produce a CV mean ± SD plot with the nominated depth marked, and a `compare_paired` verdict against the carried depth
- [ ] All figures render (none broken)
- [ ] Both `_clf` and `_reg` variable namespaces stay disjoint (no `NameError`)
- [ ] You can explain, in plain English, which depth each case carries forward and why

### Next Step:

- **Notebook 12** — Random Forests + Importance

---

---

## Bibliography

- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2023). *An Introduction to Statistical Learning with Python* — Ch. 8 (Tree-Based Methods), §8.1 Decision Trees. Springer.
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* — Chapter on tree-based methods.
- Breiman, L., Friedman, J. H., Olshen, R. A., & Stone, C. J. (1984). *Classification and Regression Trees*. Wadsworth.
- Quinlan, J. R. (1986). "Induction of decision trees." *Machine Learning*, 1(1), 81–106.
- scikit-learn User Guide — [Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
- Provost, F., & Fawcett, T. (2013). *Data Science for Business* — Chapter on segmentation and decision rules.
- Nadeau, C., & Bengio, Y. (2003). "Inference for the generalization error." *Machine Learning*, 52, 239–281. (The corrected interval in `compare_paired`.)
- Bengio, Y., & Grandvalet, Y. (2004). "No unbiased estimator of the variance of k-fold cross-validation." *Journal of Machine Learning Research*, 5, 1089–1105.
- Bates, S., Hastie, T., & Tibshirani, R. (2024). "Cross-validation: What does it estimate and how well does it do it?" *Journal of the American Statistical Association*, 119(546), 1434–1445.
- Chen, Y., & Yang, Y. (2021). "The one standard error rule for model selection: Does it work?" *Stats*, 4(4), 868–892.

---

<center>

**Thank you!**

</center>